In [21]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "pandas",
    "pyarrow",
    "fastparquet",
    "nbformat"
])

import pandas as pd
from pathlib import Path
from scipy.optimize import least_squares
from IPython.display import display
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import numpy as np
import re
from natsort import natsorted

# =============================================================================
# 1. 参数区
# =============================================================================
folder_path = Path(r"Z:\Forschung\bgA\J7099_SIE_Alterung\Studenten\csi-ych\statistik\METABatt_Sony_Murata_18650VTC6_007")
SOH_FILENAME_PATTERN = re.compile(r"BM\d+_(\d+(?:\.\d+)?)SOH\.parquet$", flags=re.IGNORECASE)


def extract_soh_from_filename(filename):
    match = SOH_FILENAME_PATTERN.search(filename.strip())
    if match is None:
        raise ValueError(f"无法从文件名解析 SOH: {filename}")
    return float(match.group(1))

# 实验设置的SOC顺序
SOC_ORDER = ["90%", "50%", "10%"]

# cycle内脉冲按照ID划分
REMOVE_PULSE_BEFORE_MIN = 60

# 实测 active 会比 3h 稍大，但不会超过 CYCLE_ACTIVE_LIMIT_HOUR.
# 因此用 CYCLE_ACTIVE_LIMIT_HOUR 作为 time_diff 判断是否进入下一 cycle 的边界。
CYCLE_ACTIVE_LIMIT_HOUR = 4.0

# 设置电流标准差
STD_LIMIT_1P5A = 0.1
STD_LIMIT_3A = 0.1

# R0线性插值设置：
# pulse段第一个测量点不稳定，因此取每个pulse段前六个点，
# 跳过第一个点后，用后五个点做线性插值/估算。
R0_PULSE_HEAD_POINTS = 6
R0_FIT_POINTS = 5
R0_TIME_AFTER_PREV_PAUSE_SEC = 0.1

OUTPUT_COLUMNS = [
    "SOH",
    "SOC",
    "File",
    "Time",
    "Current",
    "Voltage",
    "Zustand",
    "ID",
    "Zustand/Current"
]

In [22]:
# =============================================================================
# 2. 读取 parquet
# =============================================================================

parquet_files = natsorted(list(folder_path.rglob("*.parquet")))

if len(parquet_files) == 0:
    raise FileNotFoundError(f"没有在文件夹中找到 parquet 文件: {folder_path}")

df_list = []

for file in parquet_files:
    temp = pd.read_parquet(file)
    temp["File"] = file.name
    temp["SOH"] = extract_soh_from_filename(file.name)

    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

In [23]:
# =============================================================================
# 3. time diff 主函数
# =============================================================================

def build_time_diff_sequence(df):
    df_td = df.copy()

    # -----------------------------
    # 3.1 基础时间处理
    # -----------------------------
    df_td["Time"] = pd.to_datetime(df_td["Time"], utc=True, errors="coerce")
    df_td = df_td.dropna(subset=["Time", "Current", "Voltage"]).copy()
    df_td = df_td.sort_values(["File", "Time"]).reset_index(drop=True)

    # -----------------------------
    # 3.2 按 time_diff 划分 cycle / SOC
    # -----------------------------
    # 直接比较同一个 File 内相邻时间点的时间差：
    #   time_diff <= CYCLE_ACTIVE_LIMIT_HOUR：仍属于当前 cycle
    #   time_diff >  CYCLE_ACTIVE_LIMIT_HOUR：说明中间经过 pause，进入下一个 cycle
    df_td["time_diff_hour"] = (
        df_td.groupby("File")["Time"].diff() / pd.Timedelta(hours=1)
    )

    df_td["is_new_cycle"] = (
        df_td["time_diff_hour"].isna()
        | (df_td["time_diff_hour"] > CYCLE_ACTIVE_LIMIT_HOUR)
    )

    df_td["cycle_id"] = (
        df_td.groupby("File")["is_new_cycle"]
        .cumsum()
        .astype(int)
    )

    df_td["SOC"] = df_td["cycle_id"].map(
        lambda cycle_id: SOC_ORDER[(cycle_id - 1) % len(SOC_ORDER)]
    )

    cycle_start_time = df_td.groupby(["File", "cycle_id"])["Time"].transform("min")

    df_td["time_from_cycle_start_min"] = (
        df_td["Time"] - cycle_start_time
    ) / pd.Timedelta(minutes=1)

    # -----------------------------
    # 3.3 统一 Zustand
    # -----------------------------
    df_td["Zustand"] = df_td["Zustand"].astype(str)

    df_td.loc[
        df_td["Zustand"].str.startswith("DCH", na=False),
        "Zustand"
    ] = "DCH"

    df_td.loc[
        df_td["Zustand"].str.startswith("CHA", na=False),
        "Zustand"
    ] = "CHA"

    # -----------------------------
    # 3.4 生成 pulse_segment_id
    # -----------------------------
    df_td["pulse_segment_id"] = (
        df_td["File"].ne(df_td["File"].shift())
        | df_td["Zustand"].ne(df_td["Zustand"].shift())
    ).cumsum()

    # -----------------------------
    # 3.5 生成 Zustand/Current，并保留完整 time_diff_sequence
    # -----------------------------
    df_td["Zustand/Current"] = (
        df_td["Zustand"]
        + "/"
        + df_td["Current"].astype(float).round(1).astype(str)
    )

    # 这个表保留所有状态点，后面用于筛选 PAUO
    time_diff_sequence = df_td[OUTPUT_COLUMNS].copy()

    # 只保留 CHA / DCH 作为 pulse_sequence
    pulse_mask = df_td["Zustand"].str.startswith(("CHA", "DCH"), na=False)
    pulse_sequence = df_td[pulse_mask].copy()
    pulse_sequence = pulse_sequence.sort_values(
        ["File", "pulse_segment_id", "Time"]
    )

    # -----------------------------
    # 3.6 剔除电流不稳定的 pulse_segment_id
    # -----------------------------
    def is_bad_current_segment(group):
        group = group.sort_values("Time")

        current_values = group["Current"]

        # pulse段第一个测量点不稳定，因此稳定性判断也从第二个测量点开始
        current_values = group["Current"].iloc[1:]

        current_abs_level = round(current_values.abs().iloc[0], 1)
        current_std = current_values.std()

        if current_abs_level == 1.5:
            return current_std > STD_LIMIT_1P5A

        if current_abs_level == 3.0:
            return current_std > STD_LIMIT_3A

        return True

    pulse_sequence = pulse_sequence.groupby(["File", "pulse_segment_id"]).filter(
        lambda group: not is_bad_current_segment(group)
    ).copy()

    # -----------------------------
    # 3.7 用pulse段前六个点估算R0点
    # -----------------------------
    r0_rows = []

    for _, group in pulse_sequence.groupby(["File", "pulse_segment_id"], sort=False):
        group = group.sort_values("Time")

        # pulse段第一个测量点不稳定，不参与线性插值；
        # 因此先取前六个点，再用第2-6个点作为五个插值点。
        first_six_points = group.head(R0_PULSE_HEAD_POINTS)

        if len(first_six_points) < R0_PULSE_HEAD_POINTS:
            continue

        fit_points = first_six_points.iloc[1:1 + R0_FIT_POINTS].copy()

        pulse_start_time = group["Time"].iloc[0]
        file_name = group["File"].iloc[0]
        cycle_id = group["cycle_id"].iloc[0]

        previous_pause_points = df_td[
            (df_td["File"] == file_name)
            & (df_td["cycle_id"] == cycle_id)
            & (df_td["Time"] < pulse_start_time)
            & (df_td["Zustand"].str.startswith("PAU", na=False))
        ]

        if previous_pause_points.empty:
            continue

        previous_pause_end_time = previous_pause_points["Time"].max()
        r0_time = previous_pause_end_time + pd.Timedelta(seconds=R0_TIME_AFTER_PREV_PAUSE_SEC)

        # R0异常工况筛选：
        # 如果pulse段第一个测量点为0，
        # 说明该pulse_segment_id不用来进行后续计算，直接跳过。
        if group["Current"].iloc[0] == 0:
            continue

        # 构造线性拟合
        x = (fit_points["Time"] - r0_time) / pd.Timedelta(seconds=1)

        # 最小二乘线性拟合
        voltage_coef = np.polyfit(x, fit_points["Voltage"], deg=1)
        current_coef = np.polyfit(x, fit_points["Current"], deg=1)

        r0_row = fit_points.iloc[0].copy()
        r0_row["Time"] = r0_time
        
        # 计算两条直线在 x=0 时的值
        r0_row["Voltage"] = np.polyval(voltage_coef, 0)
        r0_row["Current"] = np.polyval(current_coef, 0)
        r0_row["Zustand/Current"] = (
            r0_row["Zustand"]
            + "/"
            + str(round(float(r0_row["Current"]), 1))
        )
        r0_rows.append(r0_row)

    if len(r0_rows) > 0:
        pulse_sequence = pd.DataFrame(r0_rows).reset_index(drop=True)
    else:
        pulse_sequence = pd.DataFrame(columns=OUTPUT_COLUMNS)

    # -----------------------------
    # 3.8 有效R0计算列
    # -----------------------------
    pulse_sequence = pulse_sequence[OUTPUT_COLUMNS].copy()

    return pulse_sequence, time_diff_sequence

pulse_sequence, time_diff_sequence = build_time_diff_sequence(df)

display(pulse_sequence)


,SOH,SOC,File,Time,Current,Voltage,Zustand,ID,Zustand/Current
0,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:34.850000+00:00,1.498242,4.123744,CHA,12_21,CHA/1.5
1,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:17.990000+00:00,-2.996104,4.016793,DCH,12_25,DCH/-3.0
2,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 20:56:39.010000+00:00,1.499394,3.803217,CHA,12_39,CHA/1.5
3,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 21:57:22.270000+00:00,-2.998196,3.724602,DCH,12_43,DCH/-3.0
4,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 22:58:25.640000+00:00,2.994620,3.832212,CHA,12_47,CHA/3.0
...,...,...,...,...,...,...,...,...,...
172,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 22:29:48.580000+00:00,-2.999307,3.722069,DCH,9_44,DCH/-3.0
173,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 23:30:51.900000+00:00,2.999439,3.826451,CHA,9_48,CHA/3.0
174,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 05:23:31.020000+00:00,1.497025,3.328529,CHA,9_58,CHA/1.5
175,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 06:24:14.200000+00:00,-2.992681,3.249646,DCH,9_62,DCH/-3.0


In [24]:
# =============================================================================
# 4. 根据当前筛选出来的 R0 估算点计算 R0
# =============================================================================
# pulse_sequence 是上一 cell 中 build_time_diff_sequence(df) 输出的 R0 估算点。
# 1. 找到每个 R0 估算点对应的前一个 pause 结束点
# 2. 计算电压变化量 delta_voltage
# 3. 计算电流变化量 delta_current
# 4. 计算 R0 = delta_voltage / delta_current

df_r0_base = df.copy()

# 与前面主函数保持一致：确保 Time 是可计算格式
df_r0_base["Time"] = pd.to_datetime(df_r0_base["Time"], utc=True, errors="coerce")
df_r0_base["Zustand"] = df_r0_base["Zustand"].astype(str)

# 复制当前已经计算出来的 R0 估算点
r0_result = pulse_sequence.copy()
r0_result["Time"] = pd.to_datetime(r0_result["Time"], utc=True, errors="coerce")

# 当前 R0 点的时间 = 前一个 pause 结束时间 + R0_TIME_AFTER_PREV_PAUSE_SEC
# 因此反推：
# 前一个 pause 结束时间 = 当前 R0 点时间 - R0_TIME_AFTER_PREV_PAUSE_SEC
r0_result["pause_end_time"] = (r0_result["Time"] - pd.Timedelta(seconds=R0_TIME_AFTER_PREV_PAUSE_SEC))

# 从原始数据中取出 pause 点
pause_points = df_r0_base[df_r0_base["Zustand"].str.startswith("PAU", na=False)].copy()

pause_points = pause_points[
    ["File", "Time", "Current", "Voltage"]
].rename(
    columns={
        "Time": "pause_end_time",
        "Current": "pause_end_current",
        "Voltage": "pause_end_voltage",
    }
)

# 根据 File + pause_end_time 找到每个 R0 点对应的前一个 pause 结束点
r0_result = r0_result.merge(
    pause_points,
    on=["File", "pause_end_time"],
    how="left"
)

# 计算 R0
r0_result["delta_voltage"] = (r0_result["Voltage"] - r0_result["pause_end_voltage"])
r0_result["delta_current"] = (r0_result["Current"] - r0_result["pause_end_current"])

r0_result["R0"] = np.where(
    r0_result["delta_current"] == 0,
    np.nan,
    r0_result["delta_voltage"] / r0_result["delta_current"] * 1000
)

print("当前计算出来的 R0：")
display(r0_result)


当前计算出来的 R0：


,SOH,SOC,File,Time,Current,Voltage,Zustand,ID,Zustand/Current,pause_end_time,pause_end_current,pause_end_voltage,delta_voltage,delta_current,R0
0,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:34.850000+00:00,1.498242,4.123744,CHA,12_21,CHA/1.5,2024-11-12 13:04:34.750000+00:00,0.0,4.087893,0.035851,1.498242,23.928742
1,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:17.990000+00:00,-2.996104,4.016793,DCH,12_25,DCH/-3.0,2024-11-12 14:05:17.890000+00:00,0.0,4.088227,-0.071434,-2.996104,23.842184
2,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 20:56:39.010000+00:00,1.499394,3.803217,CHA,12_39,CHA/1.5,2024-11-12 20:56:38.910000+00:00,0.0,3.776801,0.026416,1.499394,17.617473
3,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 21:57:22.270000+00:00,-2.998196,3.724602,DCH,12_43,DCH/-3.0,2024-11-12 21:57:22.170000+00:00,0.0,3.777579,-0.052978,-2.998196,17.669879
4,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 22:58:25.640000+00:00,2.994620,3.832212,CHA,12_47,CHA/3.0,2024-11-12 22:58:25.540000+00:00,0.0,3.779025,0.053187,2.994620,17.760858
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
172,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 22:29:48.580000+00:00,-2.999307,3.722069,DCH,9_44,DCH/-3.0,2024-10-23 22:29:48.480000+00:00,0.0,3.773688,-0.051619,-2.999307,17.210302
173,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 23:30:51.900000+00:00,2.999439,3.826451,CHA,9_48,CHA/3.0,2024-10-23 23:30:51.800000+00:00,0.0,3.775133,0.051317,2.999439,17.108974
174,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 05:23:31.020000+00:00,1.497025,3.328529,CHA,9_58,CHA/1.5,2024-10-24 05:23:30.920000+00:00,0.0,3.300823,0.027706,1.497025,18.507135
175,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 06:24:14.200000+00:00,-2.992681,3.249646,DCH,9_62,DCH/-3.0,2024-10-24 06:24:14.100000+00:00,0.0,3.305715,-0.056069,-2.992681,18.735434
